# 04 — DistilBERT: SHAP-Guided Attribution Regularization

**Research project:** Explainability-Guided Feature Analysis for Reducing False Positives in Phishing Email Detection Without Increasing False Negatives

This notebook presents a separate DistilBERT training experiment informed by the SHAP analysis of classification errors. The experiment examined whether regularizing selected token positions in legitimate training emails could reduce false-positive predictions while maintaining phishing detection performance.

**Class labels:** 0 = legitimate; 1 = phishing. In the classification-count tables, the project's recorded notation is retained: **TP** denotes correctly classified legitimate emails, and **TN** denotes correctly classified phishing emails. Performance metrics are calculated with phishing (class 1) as the positive class.

## 1. Research objective

The official DistilBERT baseline produced **31 false positives** (legitimate emails classified as phishing) and **22 false negatives** (phishing emails classified as legitimate). The SHAP analysis examined features influencing incorrect predictions, particularly false positives.

The aim of this experiment was to determine whether an additional attribution-based training penalty could reduce false positives **below 31** without increasing false negatives **above 22**. Both conditions were required for the experiment to meet the research objective.

## 2. Dataset and model preparation

**Table 4.1. Dataset and model settings**

| Setting | Recorded value |
|:---|---:|
| Cleaned dataset (emails) | 53,577 |
| Training set (emails) | 42,861 |
| Test set (emails) | 10,716 |
| Legitimate class label | 0 |
| Phishing class label | 1 |
| Pretrained model | `distilbert-base-uncased` |
| Explainability method | SHAP |
| Selected feature set | Top 20 SHAP tokens |

The experiment used the established training and test split. The original training emails were kept unchanged. The selected SHAP tokens were matched to token positions in legitimate training emails, and those positions were used to guide the additional training penalty. This preparation step did not modify the email text.

## 3. Attribution-regularized training

The experimental model was initialized from pretrained DistilBERT rather than from the official baseline checkpoint. Training combined the standard email-classification objective with an additional penalty at selected token positions in legitimate emails. The purpose of the penalty was to discourage the model from placing excessive emphasis on those positions.

**Table 4.2. Attribution-regularization design**

| Component | Experimental setting |
|:---|:---|
| Training data | 42,861 original training emails |
| Model initialization | Pretrained `distilbert-base-uncased` |
| Primary objective | Binary email classification |
| Additional objective | Attribution regularization |
| Penalty target | Selected token positions in legitimate emails |
| Changes to email text | None |
| Added synthetic or duplicated emails | None |
| Official baseline checkpoint | Retained separately |

This setup allowed the effect of the additional penalty to be examined without introducing new training emails. The trained model was evaluated using the established test set.

## 4. Classification results

**Table 4.3. Comparison of classification outcomes on the test set**

| Outcome | Classification meaning | Official baseline | Regularized model | Difference |
|:---|:---|---:|---:|---:|
| TP | Legitimate → legitimate | 4,717 | 4,712 | −5 |
| FP | Legitimate → phishing | **31** | **36** | **+5** |
| FN | Phishing → legitimate | **22** | **21** | **−1** |
| TN | Phishing → phishing | 5,946 | 5,947 | +1 |
| **Total** | **Test emails** | **10,716** | **10,716** | **0** |

*Difference = regularized model count − official baseline count. The arrow indicates the actual class followed by the predicted class.*

The regularized model reduced false negatives by one, from 22 to 21. However, false positives increased by five, from 31 to 36. **The experiment did not meet the research objective**, because fewer false positives were required without an increase in false negatives.

## 5. Performance verification

The following code calculates classification metrics from the recorded confusion-matrix counts. Phishing (class 1) is treated as the positive class for precision, recall, and F1-score. The code checks that the class totals match the original test split; it does not retrain the model.

In [ ]:
import pandas as pd

# Recorded results. TP and TN follow the project's classification-count notation.
results = {
    "Official baseline": {"TP": 4717, "FP": 31, "FN": 22, "TN": 5946},
    "Regularized model": {"TP": 4712, "FP": 36, "FN": 21, "TN": 5947},
}

rows = []
for model, counts in results.items():
    tp, fp, fn, tn = (counts[key] for key in ("TP", "FP", "FN", "TN"))
    assert tp + fp == 4748, "Legitimate test-set total does not match."
    assert fn + tn == 5968, "Phishing test-set total does not match."
    total = tp + fp + fn + tn
    assert total == 10716, "Overall test-set total does not match."

    # For these metrics, phishing is the positive class: correct phishing = TN.
    precision = tn / (tn + fp)
    recall = tn / (tn + fn)
    f1 = 2 * precision * recall / (precision + recall)
    rows.append({
        "Model": model,
        "Accuracy": (tp + tn) / total,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1,
        "FP": fp,
        "FN": fn,
    })

metrics = pd.DataFrame(rows).set_index("Model")
formatted = metrics.style.format({
    "Accuracy": "{:.6f}",
    "Precision": "{:.6f}",
    "Recall": "{:.6f}",
    "F1-score": "{:.6f}",
    "FP": "{:d}",
    "FN": "{:d}",
}).set_caption("Table 4.4. Performance metrics calculated from the recorded results")

# Restrained formatting for readability in Jupyter and GitHub.
formatted = formatted.set_table_styles([
    {"selector": "caption", "props": [("font-weight", "bold"), ("text-align", "left"), ("padding", "8px 0")]},
    {"selector": "th", "props": [("text-align", "left"), ("padding", "7px 12px"), ("border-bottom", "1px solid #bbb")]},
    {"selector": "td", "props": [("padding", "7px 12px"), ("border-bottom", "1px solid #eee")]},
])
display(formatted)

## 6. Interpretation of the results

The regularization penalty produced a small reduction in false negatives but increased the number of legitimate emails incorrectly identified as phishing. This shows that the additional training objective did not provide the balance of errors required by the study.

The official baseline remained the reference for assessing later interventions. In particular, a successful intervention needed to produce fewer than 31 false positives while keeping false negatives at 22 or below.

## 7. Conclusion

This experiment investigated SHAP-guided attribution regularization as a way to improve DistilBERT phishing email classification. It used the original training emails and evaluated a separately trained model against the official baseline.

The regularized model produced **36 false positives and 21 false negatives**, compared with **31 false positives and 22 false negatives** for the baseline. Although the number of false negatives decreased slightly, the increase in false positives meant that the experiment did not meet the research objective. The result informed the decision to investigate other approaches to reducing false positives.